In [1]:
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langchain_groq import ChatGroq
from langchain_core.tools import tool, InjectedToolArg
import requests
from typing import Annotated
from dotenv import load_dotenv
import json

In [2]:
load_dotenv()

llm = ChatGroq(
    model_name = "llama-3.3-70b-versatile",
    temperature=0.3
)

In [3]:
@tool
def get_conversion_rate(base_currency: str,target_currency: str) -> float:
    """This tool gives you the conversion rate of the targeted currency for your base currency"""
    url = f"https://v6.exchangerate-api.com/v6/1d33a979262a00d4db735ba8/pair/{base_currency}/{target_currency}"
    response = requests.get(url)
    return response.json()

@tool 
def convert( base_currency_value: float, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
    """On given the rate and amount, it returns the converted amount of the currency."""
    return conversion_rate * base_currency_value

In [4]:
get_conversion_rate.invoke({"base_currency": "USD","target_currency": "INR"})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1776988801,
 'time_last_update_utc': 'Fri, 24 Apr 2026 00:00:01 +0000',
 'time_next_update_unix': 1777075201,
 'time_next_update_utc': 'Sat, 25 Apr 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 94.1701}

In [5]:
llm_with_tool = llm.bind_tools([get_conversion_rate,convert])

In [6]:
messages = [HumanMessage("What is the conversion rate between INR to USD and based on that Convert 2000 Inr to dollars")]

In [7]:
ai_message = llm_with_tool.invoke(messages)
ai_message

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'ypjj7hyw6', 'function': {'arguments': '{"base_currency":"INR","target_currency":"USD"}', 'name': 'get_conversion_rate'}, 'type': 'function'}, {'id': '5waerqzh2', 'function': {'arguments': '{"base_currency_value":2000}', 'name': 'convert'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 39, 'prompt_tokens': 337, 'total_tokens': 376, 'completion_time': 0.087966891, 'completion_tokens_details': None, 'prompt_time': 0.040296568, 'prompt_tokens_details': None, 'queue_time': 0.055894442, 'total_time': 0.128263459}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019dc0d1-364d-7702-8464-4af9fdc4ce46-0', tool_calls=[{'name': 'get_conversion_rate', 'args': {'base_currency': 'INR', 'target_currency': 'USD'}, 'id': 'ypjj7hyw6', 'type': 'tool_call'}, {'name

In [8]:


tool_call = ai_message.tool_calls
print(tool_call)

#messages.append(AIMessage(ai_message))

[{'name': 'get_conversion_rate', 'args': {'base_currency': 'INR', 'target_currency': 'USD'}, 'id': 'ypjj7hyw6', 'type': 'tool_call'}, {'name': 'convert', 'args': {'base_currency_value': 2000}, 'id': '5waerqzh2', 'type': 'tool_call'}]


In [9]:
for tool_name in tool_call:
    """Execute the 1st tool to get the exchange rate"""
    if tool_name['name'] == "get_conversion_rate":
        toolmessage1 = get_conversion_rate.invoke(tool_name)
        conversion_rate = json.loads(toolmessage1.content)['conversion_rate']
        messages.append(toolmessage1)
    """Execute the 2nd tool convert the currency"""
    if tool_name['name'] == "convert":
        tool_name['args']['conversion_rate'] = conversion_rate
        toolmessage2 = convert.invoke(tool_name)
        messages.append(toolmessage2)



In [10]:

messages

[HumanMessage(content='What is the conversion rate between INR to USD and based on that Convert 2000 Inr to dollars', additional_kwargs={}, response_metadata={}),
 ToolMessage(content='{"result": "success", "documentation": "https://www.exchangerate-api.com/docs", "terms_of_use": "https://www.exchangerate-api.com/terms", "time_last_update_unix": 1776988801, "time_last_update_utc": "Fri, 24 Apr 2026 00:00:01 +0000", "time_next_update_unix": 1777075201, "time_next_update_utc": "Sat, 25 Apr 2026 00:00:01 +0000", "base_code": "INR", "target_code": "USD", "conversion_rate": 0.01062}', name='get_conversion_rate', tool_call_id='ypjj7hyw6'),
 ToolMessage(content='21.24', name='convert', tool_call_id='5waerqzh2')]

In [13]:
print(llm.invoke(messages).content)

The current conversion rate between INR (Indian Rupee) and USD (United States Dollar) is approximately 1 USD = 94.23 INR. 

To convert 2000 INR to USD, we can use the following calculation:

2000 INR / 94.23 INR/USD = approximately 21.24 USD

So, 2000 INR is equivalent to approximately 21.24 USD.
